# Online Shoppers Purchasing Intention (sujet E)

## 0. Configuration

In [20]:
import warnings
import logging

warnings.filterwarnings("ignore", message="Versions of mlflow")
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("opentelemetry").setLevel(logging.ERROR)

import os
import sys
import json
import time
import requests
import pandas as pd
from datetime import datetime
from azure.ai.ml.entities import (ManagedOnlineEndpoint, ManagedOnlineDeployment,
                                  Environment, CodeConfiguration)

In [21]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Workspace :", ml_client.workspace_name)

Workspace : mlw-iwid-dimitri


In [22]:
for d in ["src", "env", "pipeline", "deploy", "app"]:
    os.makedirs(d, exist_ok=True)
print("Dossiers prêts")

Dossiers prêts


## Étape 1 – Explorer les données

In [23]:
data_asset = ml_client.data.get(name="online_shoppers_intention", version="1")
print("Data asset :", data_asset.name, "- version", data_asset.version)

df = pd.read_csv(data_asset.path)

print("Nombre de lignes et de colonnes :", df.shape)
df.head()

Data asset : online_shoppers_intention - version 1
Nombre de lignes et de colonnes : (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


**À retenir** : 12 330 sessions, 18 colonnes, aucune valeur manquante. La cible est déséquilibrée : 85 % de sessions sans achat, 15 % avec achat.
Un modèle qui répond toujours « pas d'achat » aurait donc 85 % d'accuracy : l'accuracy n'est pas la bonne métrique ici.

In [24]:
print("Colonne cible : Revenue")
print()
print(df["Revenue"].value_counts())
print()
print((df["Revenue"].value_counts(normalize=True).round(2) * 100))
print()
print("Valeurs manquantes :", df.isna().sum().sum())

Colonne cible : Revenue

False    10422
True      1908
Name: Revenue, dtype: int64

False    85.0
True     15.0
Name: Revenue, dtype: float64

Valeurs manquantes : 0


## Étape 2 – Data asset

In [25]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

try:
    ml_client.data.get(name="online_shoppers_intention", version="2")
    print("Version 2 déjà existante, rien à créer")
except Exception:
    df_v2 = df.copy()
    df_v2["Revenue"] = df_v2["Revenue"].astype(int)
    df_v2["Weekend"] = df_v2["Weekend"].astype(int)
    df_v2 = pd.get_dummies(df_v2, columns=["Month", "VisitorType"])
    df_v2 = df_v2.rename(columns={"Revenue": "revenue"})
    df_v2.to_csv("intention_v2.csv")

    version_2 = Data(
        name="online_shoppers_intention",
        version="2",
        type=AssetTypes.URI_FILE,
        path="./intention_v2.csv",
        description="Cible en minuscules (revenue) ; booléens en 1/0 ; Month et VisitorType encodées en dummies",
    )
    ml_client.data.create_or_update(version_2)
    print("Version 2 créée")

Version 2 déjà existante, rien à créer


In [26]:
data_asset = ml_client.data.get(name="online_shoppers_intention", version="2")
print("Data asset :", data_asset.name, "- version", data_asset.version)

Data asset : online_shoppers_intention - version 2


## Étape 3 – Adapter `train.py` et lancer un premier job

In [27]:
%%writefile src/train.py

import argparse
import os
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TARGET = "revenue"

def parse_args():
    """1. Lecture des paramètres passés au script."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="Chemin du fichier CSV")
    parser.add_argument("--class_weight", type=str, default="none", choices=["none", "balanced"], help="Pondération des classes")
    parser.add_argument("--test_size", type=float, default=0.3, help="Part des données gardée pour le test")
    return parser.parse_args()

def load_data(path):
    """2. Chargement du data asset (sans la colonne d'index écrite par to_csv)."""
    df = pd.read_csv(path)
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

def train_model(X_train, y_train, class_weight):
    """3. Entraînement : standardisation + régression logistique."""
    cw = None if class_weight == "none" else class_weight
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=1000),
    )
    model.fit(X_train, y_train)
    return model

def evaluate(model, X_test, y_test):
    """4. Évaluation sur le jeu de test."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba),
    }

def main():
    args = parse_args()
    print(f"Paramètres : class_weight={args.class_weight}, test_size={args.test_size}")

    df = load_data(args.data)
    print(f"Données : {df.shape[0]} lignes, {df.shape[1]} colonnes, "
          f"taux d'achat = {df[TARGET].mean():.1%}")

    X = df.drop(columns=[TARGET]).astype(float)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=42, stratify=y)

    model = train_model(X_train, y_train, args.class_weight)
    metrics = evaluate(model, X_test, y_test)
    for name, value in metrics.items():
        print(f"{name:>10} : {value:.4f}")

    os.makedirs("outputs", exist_ok=True)
    joblib.dump(model, "outputs/model.pkl")
    print("Modèle sauvegardé dans outputs/model.pkl")

if __name__ == "__main__":
    main()

Overwriting src/train.py


In [28]:
!{sys.executable} src/train.py --data {data_asset.path} --class_weight none --test_size 0.3

Paramètres : class_weight=none, test_size=0.3
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.16.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Données : 12330 lignes, 30 colonnes, taux d'achat = 15.5%
  accuracy : 0.8792
 precision : 0.7224
    recall : 0.3549
        f1 : 0.4760
       auc : 0.8910
Modèle sauvegardé dans outputs/model.pkl


In [29]:
%%writefile env/conda.yml
name: credit-env
channels:
    - conda-forge
dependencies:
    - python=3.10
    - pip
    - pip:
        - scikit-learn==1.5.2
        - pandas==2.2.3
        - numpy==1.26.4
        - joblib==1.4.2
        - mlflow==2.22.0
        - azureml-mlflow==1.60.0

Overwriting env/conda.yml


In [30]:
env = Environment(
    name="env-credit-sklearn",
    version="3",
    description="scikit-learn 1.5.2 + MLflow pour le modèle d'intention d'achat",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file="env/conda.yml",
)

try:
    env = ml_client.environments.create_or_update(env)
except Exception:
    env = ml_client.environments.get("env-credit-sklearn", version="3")
print("Environnement :", env.name, "- version", env.version)

Environnement : env-credit-sklearn - version 3


In [31]:
PRENOM = "dimitri"
cluster_name = f"cc{PRENOM}"

In [32]:
def attendre(job):
    """Affiche le statut du job toutes les 30 secondes jusqu'à la fin."""
    while True:
        statut = str(ml_client.jobs.get(job.name).status)
        print(time.strftime("%H:%M:%S"), job.display_name, "-", statut)
        if statut.split(".")[-1].lower() in ("completed", "failed", "canceled"):
            return statut
        time.sleep(30)

In [ ]:
# Vérifications avant de lancer le job : si un assert échoue, on corrige avant d'utiliser le cluster
import importlib
from sklearn.model_selection import train_test_split

sys.path.insert(0, "src")
import train
importlib.reload(train)

# 1. Le script d'entraînement est bien écrit
assert os.path.getsize("src/train.py") > 0, "src/train.py est vide"

# 2. Les données, lues exactement comme dans train.py, sont exploitables
df_check = train.load_data(data_asset.path)
assert train.TARGET in df_check.columns, f"Colonne cible '{train.TARGET}' absente"
assert not any(c.startswith("Unnamed") for c in df_check.columns), "Une colonne d'index s'est glissée dans les variables"
assert df_check.isna().sum().sum() == 0, "Il reste des valeurs manquantes"
assert set(df_check[train.TARGET].unique()) == {0, 1}, "La cible doit valoir 0 ou 1"
assert df_check.drop(columns=[train.TARGET]).select_dtypes(exclude=["number", "bool"]).empty, \
    "Il reste des colonnes texte (Month, VisitorType ?)"

# 3. Mini entraînement local sur un échantillon : le code tourne de bout en bout
ech = df_check.sample(3000, random_state=0)
Xa, Xb, ya, yb = train_test_split(ech.drop(columns=[train.TARGET]).astype(float), ech[train.TARGET],
                                  test_size=0.3, random_state=42, stratify=ech[train.TARGET])
metriques_check = train.evaluate(train.train_model(Xa, ya, "balanced"), Xb, yb)
assert all(0 <= v <= 1 for v in metriques_check.values()), metriques_check

# 4. Les ressources Azure utilisées par le job existent
assert ml_client.compute.get(cluster_name).provisioning_state == "Succeeded", "Cluster indisponible"
ml_client.environments.get("env-credit-sklearn", version="3")
ml_client.data.get(name="online_shoppers_intention", version="2")

print(f"OK : {df_check.shape[0]} lignes, {df_check.shape[1] - 1} variables, "
      f"recall sur l'échantillon = {metriques_check['recall']:.2f}")
print("Toutes les vérifications passent : on peut lancer le job")

In [33]:
from azure.ai.ml import command, Input

job = command(
    code="./src",
    command="python train.py --data ${{inputs.data}} "
            "--class_weight ${{inputs.class_weight}} --test_size ${{inputs.test_size}}",
    inputs={
        "data": Input(type="uri_file", path="azureml:online_shoppers_intention:2"),
        "class_weight": "none",
        "test_size": 0.3,
    },
    environment="env-credit-sklearn:3",
    compute=cluster_name,
    experiment_name="online_shoppers_intention-default",
    display_name="intention-logreg-baseline",
)

returned_job = ml_client.jobs.create_or_update(job)
print("Job soumis :", returned_job.name)
print("Suivi dans le studio :", returned_job.studio_url)

Uploading src (0.0 MBs): 100%|██████████| 2915/2915 [00:00<00:00, 12128.41it/s]




Job soumis : serene_okra_q7wdqxzdfy
Suivi dans le studio : https://ml.azure.com/runs/serene_okra_q7wdqxzdfy?wsid=/subscriptions/bea7dae6-e3ec-4705-a8dc-557503a4c59a/resourcegroups/rg-iwid-zindovic/workspaces/mlw-iwid-dimitri&tid=88eebcae-d6e6-4ef7-bba4-4c34f4c2d5e0


In [34]:
attendre(returned_job)

08:42:10 intention-logreg-baseline - JobStatus.QUEUED
08:42:41 intention-logreg-baseline - JobStatus.QUEUED
08:43:11 intention-logreg-baseline - JobStatus.QUEUED
08:43:41 intention-logreg-baseline - JobStatus.QUEUED
08:44:11 intention-logreg-baseline - JobStatus.QUEUED
08:44:42 intention-logreg-baseline - JobStatus.RUNNING
08:45:12 intention-logreg-baseline - JobStatus.RUNNING
08:45:42 intention-logreg-baseline - JobStatus.RUNNING
08:46:12 intention-logreg-baseline - JobStatus.FINALIZING
08:46:43 intention-logreg-baseline - JobStatus.COMPLETED


'JobStatus.COMPLETED'

## Étape 4 – Suivre et comparer avec MLflow

In [35]:
%%writefile src/train.py

import argparse
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TARGET = "revenue"

def parse_args():
    """1. Lecture des paramètres passés au script."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="Chemin du fichier CSV")
    parser.add_argument("--class_weight", type=str, default="none", choices=["none", "balanced"], help="Pondération des classes")
    parser.add_argument("--test_size", type=float, default=0.3, help="Part des données gardée pour le test (0.3 = 30 %%)")
    return parser.parse_args()

def load_data(path):
    """2. Chargement du data asset (sans la colonne d'index écrite par to_csv)."""
    df = pd.read_csv(path)
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

def train_model(X_train, y_train, class_weight):
    """3. Entraînement : standardisation + régression logistique."""
    cw = None if class_weight == "none" else class_weight
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=1000),
    )
    model.fit(X_train, y_train)
    return model

def evaluate(model, X_test, y_test):
    """4. Évaluation sur le jeu de test."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba),
    }

def main():
    args = parse_args()
    print(f"Paramètres : class_weight={args.class_weight}, test_size={args.test_size}")

    df = load_data(args.data)

    X = df.drop(columns=[TARGET]).astype(float)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=42, stratify=y)

    with mlflow.start_run():
        mlflow.log_param("class_weight", args.class_weight)
        mlflow.log_param("test_size", args.test_size)

        model = train_model(X_train, y_train, args.class_weight)
        metrics = evaluate(model, X_test, y_test)

        for name, value in metrics.items():
            print(name, ":", round(value, 4))

        mlflow.log_metrics(metrics)

        signature = infer_signature(X_test, model.predict(X_test))
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        print("Modèle enregistré dans le run MLflow (dossier model/)")

if __name__ == "__main__":
    main()

Overwriting src/train.py


In [36]:
def lancer_job(class_weight, test_size, display_name):
    """Soumet un job d'entraînement avec les paramètres donnés."""
    job = command(
        code="./src",
        command="python train.py --data ${{inputs.data}} "
                "--class_weight ${{inputs.class_weight}} --test_size ${{inputs.test_size}}",
        inputs={
            "data": Input(type="uri_file", path="azureml:online_shoppers_intention:2"),
            "class_weight": class_weight,
            "test_size": test_size,
        },
        environment="env-credit-sklearn:3",
        compute=cluster_name,
        experiment_name="online_shoppers_intention-default",
        display_name=display_name,
    )
    return ml_client.jobs.create_or_update(job)

In [37]:
job1 = lancer_job("balanced", 0.3, "intention-mlflow-balanced-30")
job2 = lancer_job("none", 0.2, "intention-mlflow-none-20")

attendre(job1)
attendre(job2)

Uploading src (0.0 MBs): 100%|██████████| 3138/3138 [00:00<00:00, 124123.90it/s]




08:47:07 intention-mlflow-balanced-30 - JobStatus.STARTING
08:47:38 intention-mlflow-balanced-30 - JobStatus.QUEUED
08:48:08 intention-mlflow-balanced-30 - JobStatus.QUEUED
08:48:41 intention-mlflow-balanced-30 - JobStatus.COMPLETED
08:48:41 intention-mlflow-none-20 - JobStatus.COMPLETED


'JobStatus.COMPLETED'

In [38]:
import mlflow

runs = mlflow.search_runs(experiment_names=["online_shoppers_intention-default"])
cols = ["tags.mlflow.runName", "params.class_weight", "params.test_size",
        "metrics.accuracy", "metrics.precision", "metrics.recall",
        "metrics.f1", "metrics.auc"]
runs[[c for c in cols if c in runs.columns]].dropna(subset=["metrics.recall"])

,tags.mlflow.runName,params.class_weight,params.test_size,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.auc
1,intention-mlflow-balanced-30,balanced,0.3,0.854285,0.520097,0.746503,0.613065,0.900266
2,intention-mlflow-none-20,none,0.2,0.880779,0.736559,0.358639,0.482394,0.886613
5,intention-mlflow-none-20,none,0.2,0.880779,0.736559,0.358639,0.482394,0.886613


### Métrique choisie : le recall

La classe qui nous intéresse est l'**achat** (15 % des sessions). Le recall mesure la part des acheteurs réels que le modèle détecte.

- Rater un acheteur (faux négatif) = une vente ou une action marketing ciblée perdue.
- Signaler à tort un non-acheteur (faux positif) coûte peu (une promo envoyée pour rien).
- L'accuracy est trompeuse : 85 % sans rien prédire.

Avec `class_weight=none`, le recall reste vers 0,36 : le modèle rate presque 2 acheteurs sur 3.
Avec `class_weight=balanced`, il monte vers 0,75. On fixe donc le seuil à **`min_recall = 0.7`**.

## Étape 5 – Pipeline avec enregistrement conditionnel

In [39]:
%%writefile src/train.py

import argparse
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TARGET = "revenue"

def parse_args():
    """1. Lecture des paramètres passés au script."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="Chemin du fichier CSV")
    parser.add_argument("--class_weight", type=str, default="none", choices=["none", "balanced"], help="Pondération des classes")
    parser.add_argument("--test_size", type=float, default=0.3, help="Part des données gardée pour le test (0.3 = 30 %%)")
    parser.add_argument("--min_recall", type=float, default=0.5, help="Recall minimum pour enregistrer le modèle")
    parser.add_argument("--model_name", type=str, default="intention-model", help="Nom du modèle dans le registre")
    return parser.parse_args()

def load_data(path):
    """2. Chargement du data asset (sans la colonne d'index écrite par to_csv)."""
    df = pd.read_csv(path)
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

def train_model(X_train, y_train, class_weight):
    """3. Entraînement : standardisation + régression logistique."""
    cw = None if class_weight == "none" else class_weight
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=1000),
    )
    model.fit(X_train, y_train)
    return model

def evaluate(model, X_test, y_test):
    """4. Évaluation sur le jeu de test."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba),
    }

def main():
    args = parse_args()
    print(f"Paramètres : class_weight={args.class_weight}, test_size={args.test_size}")

    df = load_data(args.data)

    X = df.drop(columns=[TARGET]).astype(float)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=42, stratify=y)

    with mlflow.start_run():
        mlflow.log_param("class_weight", args.class_weight)
        mlflow.log_param("test_size", args.test_size)

        model = train_model(X_train, y_train, args.class_weight)
        metrics = evaluate(model, X_test, y_test)

        for name, value in metrics.items():
            print(name, ":", round(value, 4))

        mlflow.log_metrics(metrics)

        signature = infer_signature(X_test, model.predict(X_test))
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        print("Modèle enregistré dans le run MLflow (dossier model/)")

        if metrics["recall"] >= args.min_recall:
            run_id = mlflow.active_run().info.run_id
            mlflow.register_model(f"runs:/{run_id}/model", args.model_name)
            mlflow.set_tag("validation", "acceptée")
            print(f"Modèle validé (recall >= {args.min_recall}) : "
                  f"nouvelle version de {args.model_name}")
        else:
            mlflow.set_tag("validation", "refusée")
            print(f"Modèle refusé : recall {metrics['recall']:.3f} < {args.min_recall}")

if __name__ == "__main__":
    main()

Overwriting src/train.py


In [40]:
%%writefile pipeline/prep.py

import argparse
import os
import pandas as pd

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="CSV d'entrée")
    parser.add_argument("--output", type=str, required=True, help="Dossier de sortie")
    args = parser.parse_args()

    df = pd.read_csv(args.data, index_col=0)
    print("Lignes en entrée :", len(df))

    df = df.dropna()
    df = df.drop_duplicates()

    print("Lignes en sortie :", len(df))

    os.makedirs(args.output, exist_ok=True)
    chemin = os.path.join(args.output, "clean.csv")
    df.to_csv(chemin, index=False)
    print("Fichier écrit :", chemin)

if __name__ == "__main__":
    main()

Overwriting pipeline/prep.py


In [41]:
from azure.ai.ml import Input, Output, command

prep_step = command(
    name="prep_intention",
    display_name="1 - Préparation des données",
    code="./pipeline",
    command="python prep.py --data ${{inputs.data}} --output ${{outputs.clean_data}}",
    inputs={"data": Input(type="uri_file")},
    outputs={"clean_data": Output(type="uri_folder")},
    environment="env-credit-sklearn:3",
)

train_step = command(
    name="train_intention",
    display_name="2 - Entraînement et validation",
    code="./src",
    command="python train.py --data ${{inputs.data}}/clean.csv "
            "--class_weight ${{inputs.class_weight}} "
            "--test_size ${{inputs.test_size}} "
            "--min_recall ${{inputs.min_recall}}",
    inputs={
        "data": Input(type="uri_folder"),
        "class_weight": Input(type="string"),
        "test_size": Input(type="number"),
        "min_recall": Input(type="number"),
    },
    environment="env-credit-sklearn:3",
)

In [42]:
from azure.ai.ml.dsl import pipeline

@pipeline(description="Préparation, entraînement et validation du modèle")
def intention_pipeline(raw_data, class_weight, test_size, min_recall):
    prep_job = prep_step(data=raw_data)
    train_job = train_step(
        data=prep_job.outputs.clean_data,
        class_weight=class_weight,
        test_size=test_size,
        min_recall=min_recall,
    )

In [43]:
pipeline_job = intention_pipeline(
    raw_data=Input(type="uri_file", path="azureml:online_shoppers_intention:2"),
    class_weight="balanced",
    test_size=0.3,
    min_recall=0.7,
)

pipeline_job.settings.default_compute = cluster_name
pipeline_job.display_name = "intention_pipeline-balanced"

pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="intention-pipeline")
print(pipeline_job.studio_url)
attendre(pipeline_job)

Uploading pipeline (0.0 MBs): 100%|██████████| 1004/1004 [00:00<00:00, 8422162.43it/s]


Uploading src (0.0 MBs): 100%|██████████| 3892/3892 [00:00<00:00, 152467.44it/s]




https://ml.azure.com/runs/lemon_rhubarb_l2819cxjs5?wsid=/subscriptions/bea7dae6-e3ec-4705-a8dc-557503a4c59a/resourcegroups/rg-iwid-zindovic/workspaces/mlw-iwid-dimitri&tid=88eebcae-d6e6-4ef7-bba4-4c34f4c2d5e0
08:50:02 intention_pipeline-balanced - JobStatus.NOT_STARTED
08:50:33 intention_pipeline-balanced - JobStatus.RUNNING
08:51:03 intention_pipeline-balanced - JobStatus.RUNNING
08:51:33 intention_pipeline-balanced - JobStatus.RUNNING
08:52:03 intention_pipeline-balanced - JobStatus.COMPLETED


'JobStatus.COMPLETED'

In [44]:
modele = ml_client.models.get(name="intention-model", label="latest")
print("Dernière version :", modele.name, "version", modele.version)
print("Tags :", modele.tags)

Dernière version : intention-model version 1
Tags : {}


## Étape 6 – Déployer et tester

In [45]:
nom_modele = "intention-model"
modele = ml_client.models.get(name=nom_modele, label="latest")
print("Modèle retenu :", modele.name, "version", modele.version)

Modèle retenu : intention-model version 1


In [46]:
df_test = pd.read_csv(data_asset.path, index_col=0)

echantillon = pd.concat([df_test[df_test["revenue"] == 0].head(2),
                         df_test[df_test["revenue"] == 1].head(1)])

exemples = echantillon.drop(columns=["revenue"]).astype(float)
requete = {"data": exemples.to_dict(orient="records")}

with open("sample-request.json", "w") as f:
    json.dump(requete, f, indent=2)

print("Achat réel des 3 sessions :", echantillon["revenue"].tolist())

Achat réel des 3 sessions : [0, 0, 1]


In [47]:
%%writefile deploy/score.py

import os
import json
import pandas as pd
import mlflow.sklearn

model = None

def init():
    """Chargement du modèle au démarrage du conteneur."""
    global model
    racine = os.getenv("AZUREML_MODEL_DIR")

    chemin_modele = None
    for dossier, _, fichiers in os.walk(racine):
        if "MLmodel" in fichiers:
            chemin_modele = dossier
            break

    model = mlflow.sklearn.load_model(chemin_modele)
    print(f"[init] Modèle chargé depuis : {chemin_modele}")

def run(raw_data):
    """Appelée à chaque requête. raw_data est le texte JSON envoyé à l'API."""
    try:
        donnees = json.loads(raw_data)["data"]

        X = pd.DataFrame(donnees).astype(float)

        colonnes = getattr(model, "feature_names_in_", None)
        if colonnes is not None:
            manquantes = [c for c in colonnes if c not in X.columns]
            if manquantes:
                print(f"[run] ERREUR : colonnes manquantes {manquantes}")
                return {"erreur": f"{len(manquantes)} colonne(s) manquante(s) : {manquantes}"}
            X = X[list(colonnes)]

        predictions = model.predict(X).tolist()
        probabilites = model.predict_proba(X)[:, 1].round(3).tolist()

        print(f"[run] {len(X)} ligne(s) reçue(s), prédictions : {predictions}")
        return {"predictions": predictions, "probabilites_achat": probabilites}

    except Exception as erreur:
        print(f"[run] ERREUR : {erreur}")
        return {"erreur": str(erreur)}

Writing deploy/score.py


In [48]:
print("Dossier du notebook :", os.getcwd())
print("Contenu de deploy :", os.listdir("deploy"))
contenu = open("deploy/score.py").read()
print("ATTENTION : il reste des ______ à compléter" if "______" in contenu else "OK : score.py est complet")

Dossier du notebook : /mnt/batch/tasks/shared/LS_root/mounts/clusters/cidimitri2026/code/Users/dimitri.zindovic/tp3
Contenu de deploy : ['score.py']
OK : score.py est complet


In [49]:
%%writefile deploy/conda-deploy.yml
name: env-intention-deploy
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn==1.5.2
      - pandas==2.2.3
      - numpy==1.26.4
      - joblib==1.4.2
      - mlflow==2.22.0
      - azureml-inference-server-http==1.4.1

Writing deploy/conda-deploy.yml


In [50]:
env_deploy = Environment(
    name="env-intention-deploy",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file="deploy/conda-deploy.yml",
)
env_deploy = ml_client.environments.create_or_update(env_deploy)
print(env_deploy.name, "version", env_deploy.version)

env-intention-deploy version 1


In [ ]:
import importlib.util
import shutil

assert modele.name == nom_modele and modele.version is not None, "Modèle introuvable"
assert os.path.exists("sample-request.json"), "sample-request.json manquant"
assert "______" not in open("deploy/score.py").read(), "score.py incomplet"
assert env_deploy.version is not None, "Environnement de déploiement non créé"

dossier_local = "modele_local"
shutil.rmtree(dossier_local, ignore_errors=True)
ml_client.models.download(name=nom_modele, version=modele.version, download_path=dossier_local)
os.environ["AZUREML_MODEL_DIR"] = dossier_local

spec = importlib.util.spec_from_file_location("score", "deploy/score.py")
score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score)
score.init()
assert score.model is not None, "init() n'a pas chargé le modèle"

colonnes_modele = list(score.model.feature_names_in_)
assert list(pd.DataFrame(requete["data"]).columns) == colonnes_modele, "Colonnes de sample-request.json ≠ modèle"

reponse_locale = score.run(json.dumps(requete))
assert "predictions" in reponse_locale, f"score.py renvoie une erreur : {reponse_locale}"
assert len(reponse_locale["predictions"]) == len(requete["data"])
assert set(reponse_locale["predictions"]) <= {0, 1}

reponse_fausse = score.run(json.dumps({"data": [{"PageValues": 12}]}))
assert "erreur" in reponse_fausse, "score.py devrait signaler les colonnes manquantes"

print("Réponse locale :", reponse_locale)
print("Toutes les vérifications passent : on peut créer l'endpoint")

In [52]:
nom_endpoint = f"intention-ep-{PRENOM}-{datetime.now():%m%d%H%M}"
print("Nom de l'endpoint :", nom_endpoint)   # notez-le !

endpoint = ManagedOnlineEndpoint(name=nom_endpoint, auth_mode="key",
                                 description="API de prédiction d'achat en ligne")
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Nom de l'endpoint : intention-ep-dimitri-11050900


ManagedOnlineEndpoint({'public_network_access': <PublicNetworkAccessType.ENABLED: 'Enabled'>, 'provisioning_state': <EndpointProvisioningState.SUCCEEDED: 'Succeeded'>, 'scoring_uri': 'https://intention-ep-dimitri-11050900.spaincentral.inference.ml.azure.com/score', 'openapi_uri': 'https://intention-ep-dimitri-11050900.spaincentral.inference.ml.azure.com/swagger.json', 'name': 'intention-ep-dimitri-11050900', 'description': "API de prédiction d'achat en ligne", 'tags': {}, 'properties': {'createdBy': 'Dimitri ZINDOVIC', 'createdAt': '2026-11-05T09:00:43.578519+0000', 'lastModifiedAt': '2026-11-05T09:02:05.631554+0000', 'azureml.onlineendpointid': '/subscriptions/bea7dae6-e3ec-4705-a8dc-557503a4c59a/resourcegroups/rg-iwid-zindovic/providers/microsoft.machinelearningservices/workspaces/mlw-iwid-dimitri/onlineendpoints/intention-ep-dimitri-11050900', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/bea7dae6-e3ec-4705-a8dc-557503a4c59a/providers/Microsoft.MachineLearnin

In [53]:
deploiement = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=nom_endpoint,
    model=modele,
    environment=env_deploy,
    code_configuration=CodeConfiguration(code="deploy", scoring_script="score.py"),
    instance_type="Standard_F2s_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deploiement)

Check: endpoint intention-ep-dimitri-11050900 exists
Uploading deploy (0.0 MBs): 100%|██████████| 1753/1753 [00:00<00:00, 42060.85it/s]




..

In [54]:
from azure.core.exceptions import ResourceNotFoundError

def attendre_deploiement(nom, endpoint):
    """Affiche l'état du déploiement toutes les 30 secondes jusqu'à la fin."""
    while True:
        try:
            d = ml_client.online_deployments.get(name=nom, endpoint_name=endpoint)
            print(f"{datetime.now():%H:%M:%S} - {d.provisioning_state}")
            if d.provisioning_state in ("Succeeded", "Failed", "Canceled"):
                return d.provisioning_state
        except ResourceNotFoundError:
            print("Déploiement pas encore visible, on attend...")
        time.sleep(30)

etat = attendre_deploiement("blue", nom_endpoint)

09:02:44 - Creating
......09:03:15 - Creating
......09:03:45 - Creating
......09:04:15 - Creating
......09:04:45 - Creating
.....09:05:15 - Creating
......09:05:45 - Creating
......09:06:15 - Creating
......09:06:45 - Creating
......09:07:16 - Creating
......09:07:46 - Creating
......09:08:16 - Creating
.....09:08:46 - Creating
......09:09:16 - Creating
......09:09:46 - Creating
......09:10:16 - Creating
....09:10:46 - Succeeded


In [56]:
print(ml_client.online_deployments.get_logs(name="blue", endpoint_name=nom_endpoint, lines=40))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-11-05T09:08:41.779517Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-11-05T09:08:43.970774Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-11-05T09:09:33.915182Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-11-05T09:09:33.915182Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-11-05T09:09:33.962173Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-11-05T09:09:34.013966Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-11-05T09:09:53.999807515Z, Message: Container is ready

Container logs:
-------------

In [57]:
endpoint = ml_client.online_endpoints.get(nom_endpoint)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

endpoint = ml_client.online_endpoints.get(nom_endpoint)
print("Trafic :", endpoint.traffic)
print("URL :", endpoint.scoring_uri)

Trafic : {'blue': 100}
URL : https://intention-ep-dimitri-11050900.spaincentral.inference.ml.azure.com/score


In [58]:
reponse = ml_client.online_endpoints.invoke(endpoint_name=nom_endpoint,
                                            request_file="sample-request.json")
print(reponse)

{"predictions": [0, 0, 1], "probabilites_achat": [0.002, 0.017, 0.529]}


In [59]:
cle = ml_client.online_endpoints.get_keys(nom_endpoint).primary_key

entetes = {"Content-Type": "application/json",
           "Authorization": f"Bearer {cle}"}

r = requests.post(endpoint.scoring_uri, data=json.dumps(requete), headers=entetes)
print("Code HTTP :", r.status_code)
print("Réponse :", r.json())
print("Achat réel :", echantillon["revenue"].tolist())

Code HTTP : 200
Réponse : {'predictions': [0, 0, 1], 'probabilites_achat': [0.002, 0.017, 0.529]}
Achat réel : [0, 0, 1]


In [60]:
requete_fausse = {"data": [{"PageValues": 12}]}
r = requests.post(endpoint.scoring_uri, data=json.dumps(requete_fausse), headers=entetes)
print(r.json())

{'erreur': "27 colonne(s) manquante(s) : ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Month_Aug', 'Month_Dec', 'Month_Feb', 'Month_Jul', 'Month_June', 'Month_Mar', 'Month_May', 'Month_Nov', 'Month_Oct', 'Month_Sep', 'VisitorType_New_Visitor', 'VisitorType_Other', 'VisitorType_Returning_Visitor']"}


In [61]:
print(ml_client.online_deployments.get_logs(name="blue", endpoint_name=nom_endpoint, lines=40))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-11-05T09:08:41.779517Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-11-05T09:08:43.970774Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-11-05T09:09:33.915182Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-11-05T09:09:33.915182Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-11-05T09:09:33.962173Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-11-05T09:09:34.013966Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-11-05T09:09:53.999807515Z, Message: Container is ready

Container logs:
2026-11-05 09

## Étape 7 – Nettoyer

In [ ]:
# ml_client.online_endpoints.begin_delete(name=nom_endpoint).result()
# print("Endpoint supprimé")